In [1]:
# ============================================================
# D3 — Stage 4 Validation — Branch C: Deterministic Normalisation
# 0. Imports and frozen validation configuration
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import math
import re
import unicodedata
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

DOCUMENT_ID = "D3"
BRANCH_ID = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

EXPECTED_REFERENCE_COUNT = 70
REFERENCE_PERIOD = "May 2024"

EXPECTED_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Value",
    "Unit",
    "Reference Period"
]

REFERENCE_FIELDS = EXPECTED_FIELDS + ["Source Location"]

# Frozen from Stage 1 / D3 Branch A validation.
REFERENCE_KEY_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Reference Period"
]

ALLOWED_UNITS = {
    "workers",
    "million workers",
    "percent",
    "USD"
}

# Frozen from Branch A validation.
NUMERIC_TOLERANCE = 1e-9
FALLBACK_MIN_OCCUPATION_SIMILARITY = 0.75
FALLBACK_MIN_TOTAL_SCORE = 0.72

OUTPUT_DIR = Path("outputs_D3_validation_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("D3 Branch C validation configured.")
print("Expected Stage 1 reference records:", EXPECTED_REFERENCE_COUNT)

D3 Branch C validation configured.
Expected Stage 1 reference records: 70


In [2]:
# ------------------------------------------------------------
# 1. Upload validation inputs
# ------------------------------------------------------------
# Required:
#   1) D3_reference_values.csv
#   2) D3_branch_C_parsed_extraction.json
#   3) D3_branch_C_structure_check.json
#   4) D3_branch_C_normalisation_check.json
#
# The reference dataset supplies the fixed document-grounded truth.
# The structure check supplies technical/schema validity.
# The normalisation check supplies B→C representation-integrity evidence.
# The parsed extraction is compared without manual correction.

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [f for f in uploaded_files if f.lower().endswith(".csv")]
json_files = [f for f in uploaded_files if f.lower().endswith(".json")]

if len(csv_files) != 1:
    raise ValueError("Upload exactly one Stage 1 reference-values CSV.")

if len(json_files) != 3:
    raise ValueError(
        "Upload exactly three JSON files: parsed extraction, "
        "structure check, and normalisation check."
    )

REFERENCE_FILE = csv_files[0]
PARSED_EXTRACTION_FILE = None
STRUCTURE_CHECK_FILE = None
NORMALISATION_CHECK_FILE = None

for file_name in json_files:
    with open(file_name, "r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and "record_structure_issues" in obj
        and "schema_validity" in obj
        and "observed_record_count" in obj
    ):
        STRUCTURE_CHECK_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and obj.get("parent_branch") == PARENT_BRANCH
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_CHECK_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError("Could not identify D3 Branch C parsed extraction JSON.")

if STRUCTURE_CHECK_FILE is None:
    raise ValueError("Could not identify D3 Branch C structure-check JSON.")

if NORMALISATION_CHECK_FILE is None:
    raise ValueError("Could not identify D3 Branch C normalisation-check JSON.")

print("Reference values:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Structure check:", STRUCTURE_CHECK_FILE)
print("Normalisation check:", NORMALISATION_CHECK_FILE)

Saving D3_branch_C_parsed_extraction.json to D3_branch_C_parsed_extraction.json
Saving D3_branch_C_structure_check.json to D3_branch_C_structure_check.json
Saving D3_branch_C_normalisation_check.json to D3_branch_C_normalisation_check.json
Saving D3_reference_values.csv to D3_reference_values.csv
Reference values: D3_reference_values.csv
Parsed extraction: D3_branch_C_parsed_extraction.json
Structure check: D3_branch_C_structure_check.json
Normalisation check: D3_branch_C_normalisation_check.json


In [3]:
# ------------------------------------------------------------
# 2. Load inputs, verify identities, and preserve provenance
# ------------------------------------------------------------

with open(PARSED_EXTRACTION_FILE, "r", encoding="utf-8-sig") as f:
    extraction_json = json.load(f)

with open(STRUCTURE_CHECK_FILE, "r", encoding="utf-8-sig") as f:
    structure_check = json.load(f)

with open(NORMALISATION_CHECK_FILE, "r", encoding="utf-8-sig") as f:
    normalisation_check = json.load(f)

df_ref_raw = pd.read_csv(REFERENCE_FILE, encoding="utf-8-sig")
df_ext_raw = pd.DataFrame(extraction_json["records"])

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "structure check": structure_check,
    "normalisation check": normalisation_check
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH_ID:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )

if normalisation_check.get("parent_branch") != PARENT_BRANCH:
    raise ValueError(
        f"Unexpected Branch C parent: "
        f"{normalisation_check.get('parent_branch')}"
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256": sha256_file(PARSED_EXTRACTION_FILE),
    "structure_check_file": STRUCTURE_CHECK_FILE,
    "structure_check_sha256": sha256_file(STRUCTURE_CHECK_FILE),
    "normalisation_check_file": NORMALISATION_CHECK_FILE,
    "normalisation_check_sha256": sha256_file(NORMALISATION_CHECK_FILE)
}

print("Reference shape:", df_ref_raw.shape)
print("Extraction shape:", df_ext_raw.shape)

Reference shape: (70, 7)
Extraction shape: (74, 6)


In [4]:
# ------------------------------------------------------------
# 3. Reuse Branch C structural/schema diagnostics
# ------------------------------------------------------------
# Record-count agreement is deliberately excluded from schema validity.
# It belongs to scope/completeness, not structural conformance.

schema_validity = bool(structure_check.get("schema_validity", False))

schema_diagnostics = {
    "valid_json":
        bool(structure_check.get("valid_json", False)),

    "top_level_object_valid":
        bool(structure_check.get("top_level_object_valid", False)),

    "document_id_present":
        bool(structure_check.get("document_id_present", False)),

    "document_id_correct":
        bool(structure_check.get("document_id_correct", False)),

    "branch_present":
        bool(structure_check.get("branch_present", False)),

    "branch_correct":
        bool(structure_check.get("branch_correct", False)),

    "records_present":
        bool(structure_check.get("records_present", False)),

    "records_is_list":
        bool(structure_check.get("records_is_list", False)),

    "records_with_structure_issues":
        int(structure_check.get("records_with_structure_issues", 0)),

    "records_with_type_issues":
        int(structure_check.get("records_with_type_issues", 0)),

    "records_with_unexpected_units":
        int(structure_check.get("records_with_unexpected_units", 0)),

    "records_with_unexpected_reference_periods":
        int(structure_check.get(
            "records_with_unexpected_reference_periods", 0
        )),

    "duplicate_record_key_count":
        int(structure_check.get("duplicate_record_key_count", 0)),

    "excluded_content_issue_count":
        int(structure_check.get("excluded_content_issue_count", 0)),

    "observed_record_count":
        int(structure_check.get("observed_record_count", len(df_ext_raw))),

    "expected_record_count":
        int(structure_check.get(
            "expected_record_count", EXPECTED_REFERENCE_COUNT
        )),

    "scope_complete":
        bool(structure_check.get("scope_complete", False)),

    "schema_validity":
        schema_validity
}

print("Schema diagnostics:")
print(json.dumps(schema_diagnostics, indent=2, ensure_ascii=False))

Schema diagnostics:
{
  "valid_json": true,
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true,
  "records_with_structure_issues": 0,
  "records_with_type_issues": 0,
  "records_with_unexpected_units": 0,
  "records_with_unexpected_reference_periods": 0,
  "duplicate_record_key_count": 0,
  "excluded_content_issue_count": 0,
  "observed_record_count": 74,
  "expected_record_count": 70,
  "scope_complete": false,
  "schema_validity": true
}


In [5]:
# ------------------------------------------------------------
# 4. Reuse Branch C representation-integrity diagnostics
# ------------------------------------------------------------
# These diagnostics describe the deterministic B→C representation step.
# They are NOT extraction correctness metrics.

representation_integrity = {
    "parent_branch":
        normalisation_check.get("parent_branch"),

    "parent_equivalence_passed":
        bool(normalisation_check.get(
            "parent_equivalence_passed", False
        )),

    "normalisation_integrity_passed":
        bool(normalisation_check.get(
            "normalisation_integrity_passed", False
        )),

    "source_page_count":
        normalisation_check.get("source_page_count"),

    "all_source_page_boundaries_preserved":
        bool(normalisation_check.get(
            "all_source_page_boundaries_preserved", False
        )),

    "fixed_stage1_scope_pages":
        normalisation_check.get("fixed_stage1_scope_pages"),

    "all_scope_markers_preserved":
        bool(normalisation_check.get(
            "all_scope_markers_preserved", False
        )),

    "nonempty_line_count_preserved":
        bool(normalisation_check.get(
            "nonempty_line_count_preserved", False
        )),

    "content_line_sequence_preserved":
        bool(normalisation_check.get(
            "content_line_sequence_preserved", False
        )),

    "numeric_tokens_preserved":
        bool(normalisation_check.get(
            "numeric_tokens_preserved", False
        )),

    "semantic_label_rewriting_applied":
        bool(normalisation_check.get(
            "semantic_label_rewriting_applied", False
        )),

    "unit_semantic_remapping_applied":
        bool(normalisation_check.get(
            "unit_semantic_remapping_applied", False
        )),

    "manual_reconstruction_applied":
        bool(normalisation_check.get(
            "manual_reconstruction_applied", False
        )),

    "manual_correction_applied":
        bool(normalisation_check.get(
            "manual_correction_applied", False
        )),

    "out_of_scope_pages_removed":
        bool(normalisation_check.get(
            "out_of_scope_pages_removed", False
        )),

    "value_modification_applied":
        bool(normalisation_check.get(
            "value_modification_applied", False
        )),

    "value_rounding_applied":
        bool(normalisation_check.get(
            "value_rounding_applied", False
        )),

    "derived_calculation_applied":
        bool(normalisation_check.get(
            "derived_calculation_applied", False
        )),

    "rounded_million_values_expanded":
        bool(normalisation_check.get(
            "rounded_million_values_expanded", False
        )),

    "reference_values_used_for_transformation":
        bool(normalisation_check.get(
            "reference_values_used_for_transformation", False
        ))
}

print("Branch C representation-integrity diagnostics:")
print(json.dumps(representation_integrity, indent=2, ensure_ascii=False))

if not representation_integrity["normalisation_integrity_passed"]:
    print(
        "WARNING: Branch C normalisation integrity did not pass. "
        "Extraction validation can still describe the observed response, "
        "but Stage 5 interpretation must distinguish representation loss "
        "from LLM extraction error."
    )

if representation_integrity["reference_values_used_for_transformation"]:
    raise ValueError(
        "Reference values were reportedly used during Branch C "
        "transformation, violating the experimental design."
    )

Branch C representation-integrity diagnostics:
{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "source_page_count": 23,
  "all_source_page_boundaries_preserved": true,
  "fixed_stage1_scope_pages": [
    1,
    5
  ],
  "all_scope_markers_preserved": true,
  "nonempty_line_count_preserved": true,
  "content_line_sequence_preserved": true,
  "numeric_tokens_preserved": true,
  "semantic_label_rewriting_applied": false,
  "unit_semantic_remapping_applied": false,
  "manual_reconstruction_applied": false,
  "manual_correction_applied": false,
  "out_of_scope_pages_removed": false,
  "value_modification_applied": false,
  "value_rounding_applied": false,
  "derived_calculation_applied": false,
  "rounded_million_values_expanded": false,
  "reference_values_used_for_transformation": false
}


In [6]:
# ------------------------------------------------------------
# 5. Verify Stage 1 reference and extraction fields
# ------------------------------------------------------------

missing_reference_fields = [
    field for field in REFERENCE_FIELDS
    if field not in df_ref_raw.columns
]

if missing_reference_fields:
    raise ValueError(
        f"Stage 1 reference dataset is missing fields: "
        f"{missing_reference_fields}"
    )

if len(df_ref_raw) != EXPECTED_REFERENCE_COUNT:
    raise ValueError(
        f"Unexpected Stage 1 reference count: {len(df_ref_raw)} "
        f"(expected {EXPECTED_REFERENCE_COUNT})."
    )

missing_extraction_columns = [
    field for field in EXPECTED_FIELDS
    if field not in df_ext_raw.columns
]

# Preserve the raw parsed extraction.
# Comparison is performed on a separate copy.
df_ext = df_ext_raw.copy()

# Missing fields are introduced as null only in the validation copy.
# Schema validity remains determined by the Branch C structure check.
for field in missing_extraction_columns:
    df_ext[field] = np.nan

df_ref = df_ref_raw[REFERENCE_FIELDS].copy()
df_ext = df_ext[EXPECTED_FIELDS].copy()

print("Reference records:", len(df_ref))
print("Extracted records:", len(df_ext))
print("Missing extraction columns:", missing_extraction_columns)

Reference records: 70
Extracted records: 74
Missing extraction columns: []


In [7]:
# ------------------------------------------------------------
# 6. Controlled comparison normalisation
# ------------------------------------------------------------
# FROZEN FROM D3 BRANCH A/B VALIDATION.
#
# IMPORTANT:
# This is Stage 4 comparison-only normalisation.
# It is separate from Branch C input normalisation and never
# modifies the preserved Branch C extraction.

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = (
        text
        .replace("\u00a0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def normalize_number(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, bool):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()

    if not text:
        return np.nan

    text = (
        text
        .replace("$", "")
        .replace("%", "")
        .replace(",", "")
        .replace("\u00a0", "")
        .replace(" ", "")
    )

    if text.startswith("(") and text.endswith(")"):
        text = "-" + text[1:-1]

    try:
        return float(text)
    except (TypeError, ValueError):
        return np.nan


def numbers_match(reference_value, extracted_value):
    ref_num = normalize_number(reference_value)
    ext_num = normalize_number(extracted_value)

    if pd.isna(ref_num) and pd.isna(ext_num):
        return True

    if pd.isna(ref_num) or pd.isna(ext_num):
        return False

    return math.isclose(
        ref_num,
        ext_num,
        rel_tol=0.0,
        abs_tol=NUMERIC_TOLERANCE
    )


def text_similarity(left, right):
    left = normalize_text(left)
    right = normalize_text(right)

    if not left and not right:
        return 1.0

    if not left or not right:
        return 0.0

    if left == right:
        return 1.0

    sequence_score = SequenceMatcher(
        None,
        left,
        right
    ).ratio()

    left_tokens = set(
        re.findall(r"[a-z0-9]+", left)
    )

    right_tokens = set(
        re.findall(r"[a-z0-9]+", right)
    )

    if left_tokens or right_tokens:
        token_score = (
            len(left_tokens & right_tokens)
            / len(left_tokens | right_tokens)
        )
    else:
        token_score = 0.0

    return max(
        sequence_score,
        token_score
    )


def occupation_similarity(left, right):
    left_norm = normalize_text(left)
    right_norm = normalize_text(right)

    score = text_similarity(
        left_norm,
        right_norm
    )

    if (
        left_norm
        and right_norm
        and (
            left_norm in right_norm
            or right_norm in left_norm
        )
    ):
        score = max(score, 0.95)

    return score

In [8]:
# ------------------------------------------------------------
# 7. Verify uniqueness of the fixed Stage 1 identity
# ------------------------------------------------------------
# FROZEN FROM D3 BRANCH A VALIDATION.

def strict_identity_key(row):
    return tuple(
        normalize_text(row[field])
        for field in REFERENCE_KEY_FIELDS
    )

df_ref["_strict_key"] = df_ref.apply(
    strict_identity_key,
    axis=1
)

df_ext["_strict_key"] = df_ext.apply(
    strict_identity_key,
    axis=1
)

reference_duplicate_count = int(
    df_ref["_strict_key"].duplicated(
        keep=False
    ).sum()
)

if reference_duplicate_count > 0:
    raise ValueError(
        "The fixed Stage 1 reference identity is not unique."
    )

extraction_duplicate_identity_rows = int(
    df_ext["_strict_key"].duplicated(
        keep=False
    ).sum()
)

print(
    "Reference duplicate identity rows:",
    reference_duplicate_count
)
print(
    "Extraction duplicate identity rows:",
    extraction_duplicate_identity_rows
)

Reference duplicate identity rows: 0
Extraction duplicate identity rows: 0


In [9]:
# ------------------------------------------------------------
# 8. One-to-one record alignment
# ------------------------------------------------------------
# FROZEN FROM D3 BRANCH A/B VALIDATION.
#
# Value and Unit are deliberately excluded from alignment.
# Otherwise correct numerical content could influence the decision
# about whether an extracted row represents an expected observation.

used_extraction = set()
matched_pairs = []

# ---- Stage 1: exact normalised descriptive identity ----
extraction_key_index = {}

for ext_idx, key in df_ext["_strict_key"].items():
    extraction_key_index.setdefault(
        key,
        []
    ).append(ext_idx)

for ref_idx, ref_row in df_ref.iterrows():

    candidates = [
        idx
        for idx in extraction_key_index.get(
            ref_row["_strict_key"],
            []
        )
        if idx not in used_extraction
    ]

    if candidates:
        ext_idx = candidates[0]

        used_extraction.add(
            ext_idx
        )

        matched_pairs.append({
            "reference_index":
                ref_idx,

            "extraction_index":
                ext_idx,

            "match_method":
                "strict_identity",

            "alignment_score":
                1.0
        })


matched_reference = {
    pair["reference_index"]
    for pair in matched_pairs
}


# ---- Stage 2: controlled descriptive fallback ----
# Same Reference Period is required.
candidate_pairs = []

for ref_idx, ref_row in df_ref.iterrows():

    if ref_idx in matched_reference:
        continue

    for ext_idx, ext_row in df_ext.iterrows():

        if ext_idx in used_extraction:
            continue

        if (
            normalize_text(
                ref_row["Reference Period"]
            )
            !=
            normalize_text(
                ext_row["Reference Period"]
            )
        ):
            continue

        section_score = text_similarity(
            ref_row["Section"],
            ext_row["Section"]
        )

        indicator_score = text_similarity(
            ref_row["Indicator"],
            ext_row["Indicator"]
        )

        occupation_score = occupation_similarity(
            ref_row["Occupation or Group"],
            ext_row["Occupation or Group"]
        )

        total_score = (
            0.20 * section_score
            + 0.25 * indicator_score
            + 0.55 * occupation_score
        )

        if (
            occupation_score
            >= FALLBACK_MIN_OCCUPATION_SIMILARITY
            and total_score
            >= FALLBACK_MIN_TOTAL_SCORE
        ):
            candidate_pairs.append({
                "reference_index":
                    ref_idx,

                "extraction_index":
                    ext_idx,

                "alignment_score":
                    total_score,

                "section_score":
                    section_score,

                "indicator_score":
                    indicator_score,

                "occupation_score":
                    occupation_score
            })


# Deterministic greedy one-to-one assignment:
# strongest descriptive pairs first.
candidate_pairs.sort(
    key=lambda x: (
        -x["alignment_score"],
        -x["occupation_score"],
        x["reference_index"],
        x["extraction_index"]
    )
)

for candidate in candidate_pairs:

    ref_idx = candidate[
        "reference_index"
    ]

    ext_idx = candidate[
        "extraction_index"
    ]

    if (
        ref_idx in matched_reference
        or ext_idx in used_extraction
    ):
        continue

    matched_reference.add(
        ref_idx
    )

    used_extraction.add(
        ext_idx
    )

    matched_pairs.append({
        **candidate,
        "match_method":
            "descriptive_fallback"
    })


matched_pairs = sorted(
    matched_pairs,
    key=lambda x: x["reference_index"]
)

matched_reference_indices = {
    pair["reference_index"]
    for pair in matched_pairs
}

missing_reference_indices = [
    idx
    for idx in df_ref.index
    if idx not in matched_reference_indices
]

unsupported_extraction_indices = [
    idx
    for idx in df_ext.index
    if idx not in used_extraction
]

print("Aligned records:", len(matched_pairs))
print("Missing expected records:", len(missing_reference_indices))
print("Unsupported extracted records:", len(unsupported_extraction_indices))

match_method_counts = pd.Series(
    [
        pair["match_method"]
        for pair in matched_pairs
    ],
    dtype="object"
).value_counts()

print("\nMatch methods:")
print(match_method_counts)

Aligned records: 69
Missing expected records: 1
Unsupported extracted records: 5

Match methods:
strict_identity         40
descriptive_fallback    29
Name: count, dtype: int64


In [10]:
# ------------------------------------------------------------
# 9. Compare all requested fields after alignment
# ------------------------------------------------------------

record_rows = []
field_rows = []

for pair in matched_pairs:

    ref = df_ref.loc[
        pair["reference_index"]
    ]

    ext = df_ext.loc[
        pair["extraction_index"]
    ]

    field_matches = {
        "Section":
            normalize_text(
                ref["Section"]
            )
            ==
            normalize_text(
                ext["Section"]
            ),

        "Indicator":
            normalize_text(
                ref["Indicator"]
            )
            ==
            normalize_text(
                ext["Indicator"]
            ),

        "Occupation or Group":
            normalize_text(
                ref["Occupation or Group"]
            )
            ==
            normalize_text(
                ext["Occupation or Group"]
            ),

        "Value":
            numbers_match(
                ref["Value"],
                ext["Value"]
            ),

        "Unit":
            normalize_text(
                ref["Unit"]
            )
            ==
            normalize_text(
                ext["Unit"]
            ),

        "Reference Period":
            normalize_text(
                ref["Reference Period"]
            )
            ==
            normalize_text(
                ext["Reference Period"]
            )
    }

    all_fields_match = all(
        field_matches.values()
    )

    label_fields_match = all([
        field_matches["Section"],
        field_matches["Indicator"],
        field_matches["Occupation or Group"]
    ])

    content_fields_match = all([
        field_matches["Value"],
        field_matches["Unit"],
        field_matches["Reference Period"]
    ])

    if all_fields_match:
        status = "fully_correct"

    elif (
        content_fields_match
        and not label_fields_match
    ):
        status = "label_only_discrepancy"

    else:
        status = "discrepant"

    mismatched_fields = [
        field
        for field, is_match
        in field_matches.items()
        if not is_match
    ]

    record_rows.append({
        "Reference Record ID":
            f"D3-REF-{pair['reference_index'] + 1:03d}",

        "Extraction Record ID":
            f"D3-C-{pair['extraction_index'] + 1:03d}",

        "Section_ref":
            ref["Section"],

        "Section_ext":
            ext["Section"],

        "Indicator_ref":
            ref["Indicator"],

        "Indicator_ext":
            ext["Indicator"],

        "Occupation or Group_ref":
            ref["Occupation or Group"],

        "Occupation or Group_ext":
            ext["Occupation or Group"],

        "Value_ref":
            ref["Value"],

        "Value_ext":
            ext["Value"],

        "Unit_ref":
            ref["Unit"],

        "Unit_ext":
            ext["Unit"],

        "Reference Period_ref":
            ref["Reference Period"],

        "Reference Period_ext":
            ext["Reference Period"],

        "Source Location":
            ref["Source Location"],

        "Match Method":
            pair["match_method"],

        "Alignment Score":
            pair.get("alignment_score"),

        "Section_match":
            field_matches["Section"],

        "Indicator_match":
            field_matches["Indicator"],

        "Occupation or Group_match":
            field_matches["Occupation or Group"],

        "Value_match":
            field_matches["Value"],

        "Unit_match":
            field_matches["Unit"],

        "Reference Period_match":
            field_matches["Reference Period"],

        "all_fields_match":
            all_fields_match,

        "record_status":
            status,

        "mismatched_fields":
            ", ".join(
                mismatched_fields
            )
    })

    for field in EXPECTED_FIELDS:

        field_rows.append({
            "Reference Record ID":
                f"D3-REF-{pair['reference_index'] + 1:03d}",

            "Extraction Record ID":
                f"D3-C-{pair['extraction_index'] + 1:03d}",

            "Field":
                field,

            "Field Match":
                field_matches[field],

            "Match Method":
                pair["match_method"],

            "Source Location":
                ref["Source Location"]
        })


record_validation_df = pd.DataFrame(
    record_rows
)

field_validation_df = pd.DataFrame(
    field_rows
)

display(
    record_validation_df.head()
)

,Reference Record ID,Extraction Record ID,Section_ref,Section_ext,Indicator_ref,Indicator_ext,Occupation or Group_ref,Occupation or Group_ext,Value_ref,Value_ext,...,Alignment Score,Section_match,Indicator_match,Occupation or Group_match,Value_match,Unit_match,Reference Period_match,all_fields_match,record_status,mismatched_fields
0,D3-REF-001,D3-C-001,Production occupations,OCCUPATIONAL EMPLOYMENT AND WAGES - MAY 2024,Employment,Employment,Production occupations,Production occupations,8.7,8.7,...,0.866667,False,True,True,True,True,True,False,label_only_discrepancy,Section
1,D3-REF-002,D3-C-002,Production occupations,OCCUPATIONAL EMPLOYMENT AND WAGES - MAY 2024,Share of national employment,Employment share,Production occupations,Production occupations,5.7,5.7,...,0.741667,False,False,True,True,True,True,False,label_only_discrepancy,"Section, Indicator"
2,D3-REF-003,D3-C-007,Production occupations,Production occupations,Employment,Employment,miscellaneous assemblers and fabricators,miscellaneous assemblers and fabricators,1.5,1.5,...,1.000000,True,True,True,True,True,True,True,fully_correct,
3,D3-REF-004,D3-C-008,Production occupations,Production occupations,Employment,Employment,first-line supervisors of production and opera...,first-line supervisors of production and opera...,685140.0,685140.0,...,1.000000,True,True,True,True,True,True,True,fully_correct,
4,D3-REF-005,D3-C-009,Production occupations,Production occupations,Employment,Employment,"inspectors, testers, sorters, samplers, and we...","inspectors, testers, sorters, samplers, and we...",591180.0,591180.0,...,1.000000,True,True,True,True,True,True,True,fully_correct,


In [11]:
# ------------------------------------------------------------
# 10. Classify missing, unsupported, and discrepancy outcomes
# ------------------------------------------------------------

missing_records = df_ref.loc[
    missing_reference_indices
].copy()

unsupported_records = df_ext.loc[
    unsupported_extraction_indices
].copy()

if "_strict_key" in missing_records.columns:
    missing_records = missing_records.drop(
        columns=["_strict_key"]
    )

if "_strict_key" in unsupported_records.columns:
    unsupported_records = unsupported_records.drop(
        columns=["_strict_key"]
    )

fully_correct_records = record_validation_df[
    record_validation_df["record_status"]
    == "fully_correct"
].copy()

label_only_discrepancies = record_validation_df[
    record_validation_df["record_status"]
    == "label_only_discrepancy"
].copy()

other_discrepancies = record_validation_df[
    record_validation_df["record_status"]
    == "discrepant"
].copy()

all_discrepancies = record_validation_df[
    record_validation_df[
        "record_status"
    ].isin([
        "label_only_discrepancy",
        "discrepant"
    ])
].copy()

print(
    "Fully correct:",
    len(fully_correct_records)
)
print(
    "Label-only discrepancies:",
    len(label_only_discrepancies)
)
print(
    "Other discrepancies:",
    len(other_discrepancies)
)
print(
    "Missing:",
    len(missing_records)
)
print(
    "Unsupported:",
    len(unsupported_records)
)

Fully correct: 40
Label-only discrepancies: 29
Other discrepancies: 0
Missing: 1
Unsupported: 5


In [12]:
# ------------------------------------------------------------
# 11. Calculate common validation metrics
# ------------------------------------------------------------

N_REF = int(
    len(df_ref)
)

N_EXT = int(
    len(df_ext)
)

N_ALIGNED = int(
    len(record_validation_df)
)

N_CORRECT = int(
    len(fully_correct_records)
)

N_LABEL_ONLY = int(
    len(label_only_discrepancies)
)

N_DISCREPANT = int(
    len(all_discrepancies)
)

N_MISSING = int(
    len(missing_records)
)

N_UNSUPPORTED = int(
    len(unsupported_records)
)

# Scope completeness is independent of field correctness.
completeness = (
    N_ALIGNED / N_REF
    if N_REF else 0.0
)

missing_rate = (
    N_MISSING / N_REF
    if N_REF else 0.0
)

# Exact record-level metrics:
# only fully correct aligned records count as correct.
record_precision = (
    N_CORRECT / N_EXT
    if N_EXT else 0.0
)

record_recall = (
    N_CORRECT / N_REF
    if N_REF else 0.0
)

record_f1 = (
    2 * record_precision * record_recall
    / (record_precision + record_recall)
    if (record_precision + record_recall)
    else 0.0
)

hallucination_rate = (
    N_UNSUPPORTED / N_EXT
    if N_EXT else 0.0
)

discrepancy_rate = (
    N_DISCREPANT / N_ALIGNED
    if N_ALIGNED else 0.0
)

field_accuracy_among_aligned = {}

for field in EXPECTED_FIELDS:

    rows = field_validation_df[
        field_validation_df["Field"]
        == field
    ]

    field_accuracy_among_aligned[field] = (
        float(
            rows["Field Match"].mean()
        )
        if len(rows)
        else 0.0
    )

correct_field_instances = int(
    field_validation_df[
        "Field Match"
    ].sum()
)

expected_field_instances = int(
    N_REF
    * len(EXPECTED_FIELDS)
)

# Missing expected records therefore contribute incorrect field instances.
overall_field_accuracy = (
    correct_field_instances
    / expected_field_instances
    if expected_field_instances
    else 0.0
)

print("Reference records:", N_REF)
print("Extracted records:", N_EXT)
print("Aligned records:", N_ALIGNED)
print("Fully correct:", N_CORRECT)
print("Label-only discrepancies:", N_LABEL_ONLY)
print("Total discrepant:", N_DISCREPANT)
print("Missing:", N_MISSING)
print("Unsupported:", N_UNSUPPORTED)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1, 4))
print("Overall field accuracy:", round(overall_field_accuracy, 4))

Reference records: 70
Extracted records: 74
Aligned records: 69
Fully correct: 40
Label-only discrepancies: 29
Total discrepant: 29
Missing: 1
Unsupported: 5
Completeness: 0.9857
Exact F1: 0.5556
Overall field accuracy: 0.9048


In [13]:
# ------------------------------------------------------------
# 12. Field-level error summary
# ------------------------------------------------------------

field_error_summary = []

for field in EXPECTED_FIELDS:

    rows = field_validation_df[
        field_validation_df["Field"]
        == field
    ]

    correct_aligned = int(
        rows["Field Match"].sum()
    )

    incorrect_aligned = int(
        len(rows)
        - correct_aligned
    )

    field_error_summary.append({
        "field":
            field,

        "used_in_strict_identity":
            field in REFERENCE_KEY_FIELDS,

        "aligned_records_evaluated":
            int(len(rows)),

        "correct_values_among_aligned":
            correct_aligned,

        "incorrect_values_among_aligned":
            incorrect_aligned,

        "accuracy_among_aligned":
            (
                round(
                    correct_aligned
                    / len(rows),
                    4
                )
                if len(rows)
                else 0.0
            ),

        "missing_expected_instances":
            N_MISSING,

        "overall_correct_instances":
            correct_aligned,

        "overall_expected_instances":
            N_REF,

        "overall_field_accuracy":
            (
                round(
                    correct_aligned
                    / N_REF,
                    4
                )
                if N_REF
                else 0.0
            )
    })

field_error_summary_df = pd.DataFrame(
    field_error_summary
)

display(
    field_error_summary_df
)

,field,used_in_strict_identity,aligned_records_evaluated,correct_values_among_aligned,incorrect_values_among_aligned,accuracy_among_aligned,missing_expected_instances,overall_correct_instances,overall_expected_instances,overall_field_accuracy
0,Section,True,69,65,4,0.9420,1,65,70,0.9286
1,Indicator,True,69,49,20,0.7101,1,49,70,0.7000
2,Occupation or Group,True,69,59,10,0.8551,1,59,70,0.8429
3,Value,False,69,69,0,1.0000,1,69,70,0.9857
4,Unit,False,69,69,0,1.0000,1,69,70,0.9857
5,Reference Period,True,69,69,0,1.0000,1,69,70,0.9857


In [14]:
# ------------------------------------------------------------
# 13. Section-level performance
# ------------------------------------------------------------

section_rows = []

for section, ref_group in df_ref.groupby(
    "Section",
    dropna=False
):

    ref_indices = set(
        ref_group.index
    )

    aligned_section = record_validation_df[
        record_validation_df[
            "Reference Record ID"
        ].apply(
            lambda value:
                int(
                    str(value).split("-")[-1]
                ) - 1
                in ref_indices
        )
    ]

    section_rows.append({
        "Section":
            section,

        "Reference Records":
            len(ref_group),

        "Aligned Records":
            len(aligned_section),

        "Fully Correct Records":
            int(
                (
                    aligned_section["record_status"]
                    == "fully_correct"
                ).sum()
            ),

        "Label-only Discrepancies":
            int(
                (
                    aligned_section["record_status"]
                    == "label_only_discrepancy"
                ).sum()
            ),

        "Other Discrepancies":
            int(
                (
                    aligned_section["record_status"]
                    == "discrepant"
                ).sum()
            ),

        "Missing Records":
            len(ref_group)
            - len(aligned_section)
    })

section_performance_df = pd.DataFrame(
    section_rows
)

display(
    section_performance_df
)

,Section,Reference Records,Aligned Records,Fully Correct Records,Label-only Discrepancies,Other Discrepancies,Missing Records
0,Architecture and engineering occupations,18,17,11,6,0,1
1,Building and grounds cleaning and maintenance ...,14,14,12,2,0,0
2,Largest occupations,7,7,7,0,0,0
3,Production occupations,23,23,10,13,0,0
4,Public sector occupations,8,8,0,8,0,0


In [15]:
# ------------------------------------------------------------
# 14. Build Branch C validation summary
# ------------------------------------------------------------

summary = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH_ID,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "label_only_discrepant_records":
        N_LABEL_ONLY,

    "discrepant_records_total":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    # Dissertation-facing term.
    "unsupported_records":
        N_UNSUPPORTED,

    # Alias retained for continuity with earlier code/metrics.
    "hallucinated_records":
        N_UNSUPPORTED,

    "completeness":
        round(
            completeness,
            4
        ),

    "missing_rate":
        round(
            missing_rate,
            4
        ),

    "record_precision_exact":
        round(
            record_precision,
            4
        ),

    "record_recall_exact":
        round(
            record_recall,
            4
        ),

    "record_f1_exact":
        round(
            record_f1,
            4
        ),

    "hallucination_rate":
        round(
            hallucination_rate,
            4
        ),

    "discrepancy_rate_among_aligned":
        round(
            discrepancy_rate,
            4
        ),

    "overall_field_accuracy":
        round(
            overall_field_accuracy,
            4
        ),

    "field_accuracy_among_aligned": {
        key:
            round(
                value,
                4
            )
        for key, value
        in field_accuracy_among_aligned.items()
    },

    "schema_validity":
        schema_validity,

    "schema_diagnostics":
        schema_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "reference_identity_fields":
        REFERENCE_KEY_FIELDS,

    "comparison_rules_frozen_from_branch_A":
        True,

    "reference_dataset_branch_independent":
        True,

    "alignment_rules": {
        "strict_identity_first":
            True,

        "fallback_uses_descriptive_fields_only":
            True,

        "value_used_for_alignment":
            False,

        "unit_used_for_alignment":
            False,

        "fallback_min_occupation_similarity":
            FALLBACK_MIN_OCCUPATION_SIMILARITY,

        "fallback_min_total_score":
            FALLBACK_MIN_TOTAL_SCORE,

        "alignment_weights": {
            "Section":
                0.20,
            "Indicator":
                0.25,
            "Occupation or Group":
                0.55
        }
    },

    "comparison_rules": {
        "text":
            (
                "Unicode NFKC, apostrophe/dash standardisation, "
                "whitespace collapse and case folding"
            ),

        "unit":
            (
                "Text normalisation only; "
                "no semantic unit remapping"
            ),

        "numeric":
            (
                "Direct numerical comparison "
                "in the reported source scale"
            ),

        "numeric_tolerance":
            NUMERIC_TOLERANCE,

        "rounded_million_values_expanded":
            False,

        "controlled_branch_C_specific_aliases_added":
            False
    },

    "normalisation_note":
        (
            "Stage 4 comparison normalisation is applied only to "
            "comparison copies using rules frozen in D3 Branch A. "
            "It is distinct from Branch C deterministic input "
            "normalisation; the preserved extraction is not modified."
        ),

    "input_provenance":
        input_provenance
}

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D3",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "reference_records": 70,
  "extracted_records": 74,
  "aligned_records": 69,
  "fully_correct_records": 40,
  "label_only_discrepant_records": 29,
  "discrepant_records_total": 29,
  "missing_records": 1,
  "unsupported_records": 5,
  "hallucinated_records": 5,
  "completeness": 0.9857,
  "missing_rate": 0.0143,
  "record_precision_exact": 0.5405,
  "record_recall_exact": 0.5714,
  "record_f1_exact": 0.5556,
  "hallucination_rate": 0.0676,
  "discrepancy_rate_among_aligned": 0.4203,
  "overall_field_accuracy": 0.9048,
  "field_accuracy_among_aligned": {
    "Section": 0.942,
    "Indicator": 0.7101,
    "Occupation or Group": 0.8551,
    "Value": 1.0,
    "Unit": 1.0,
    "Reference Period": 1.0
  },
  "schema_validity": true,
  "schema_diagnostics": {
    "valid_json": true,
    "top_level_object_valid": true,
    "document_id_present": true,
    "document_id_correct": t

In [16]:
# ------------------------------------------------------------
# 15. Compact overall-results table
# ------------------------------------------------------------

overall_metrics_df = pd.DataFrame([
    {
        "metric":
            "Reference records",
        "value":
            N_REF
    },
    {
        "metric":
            "Extracted records",
        "value":
            N_EXT
    },
    {
        "metric":
            "Aligned records",
        "value":
            N_ALIGNED
    },
    {
        "metric":
            "Fully correct records",
        "value":
            N_CORRECT
    },
    {
        "metric":
            "Label-only discrepant records",
        "value":
            N_LABEL_ONLY
    },
    {
        "metric":
            "Total discrepant records",
        "value":
            N_DISCREPANT
    },
    {
        "metric":
            "Missing records",
        "value":
            N_MISSING
    },
    {
        "metric":
            "Unsupported records",
        "value":
            N_UNSUPPORTED
    },
    {
        "metric":
            "Completeness",
        "value":
            round(completeness, 4)
    },
    {
        "metric":
            "Exact precision",
        "value":
            round(record_precision, 4)
    },
    {
        "metric":
            "Exact recall",
        "value":
            round(record_recall, 4)
    },
    {
        "metric":
            "Exact F1",
        "value":
            round(record_f1, 4)
    },
    {
        "metric":
            "Overall field accuracy",
        "value":
            round(overall_field_accuracy, 4)
    },
    {
        "metric":
            "Hallucination/unsupported rate",
        "value":
            round(hallucination_rate, 4)
    },
    {
        "metric":
            "Schema validity",
        "value":
            schema_validity
    },
    {
        "metric":
            "Branch C normalisation integrity",
        "value":
            representation_integrity[
                "normalisation_integrity_passed"
            ]
    }
])

display(
    overall_metrics_df
)

,metric,value
0,Reference records,70
1,Extracted records,74
2,Aligned records,69
3,Fully correct records,40
4,Label-only discrepant records,29
5,Total discrepant records,29
6,Missing records,1
7,Unsupported records,5
8,Completeness,0.9857
9,Exact precision,0.5405


In [17]:
# ------------------------------------------------------------
# 16. Validation integrity checks
# ------------------------------------------------------------

# Every fixed reference observation is aligned or missing.
assert (
    N_ALIGNED
    + N_MISSING
    == N_REF
)

# Every extracted observation is aligned or unsupported.
assert (
    N_ALIGNED
    + N_UNSUPPORTED
    == N_EXT
)

# Every aligned observation is fully correct or discrepant.
assert (
    N_CORRECT
    + N_DISCREPANT
    == N_ALIGNED
)

# Label-only discrepancies are a subset of total discrepancies.
assert (
    N_LABEL_ONLY
    <= N_DISCREPANT
)

# Fixed Stage 1 strict identity must remain unique.
assert (
    reference_duplicate_count
    == 0
)

for metric_name, metric_value in {
    "completeness":
        completeness,

    "missing_rate":
        missing_rate,

    "record_precision":
        record_precision,

    "record_recall":
        record_recall,

    "record_f1":
        record_f1,

    "hallucination_rate":
        hallucination_rate,

    "discrepancy_rate":
        discrepancy_rate,

    "overall_field_accuracy":
        overall_field_accuracy
}.items():

    assert (
        0.0
        <= metric_value
        <= 1.0
    ), (
        f"Invalid {metric_name}: "
        f"{metric_value}"
    )

print(
    "Validation integrity checks passed."
)

Validation integrity checks passed.


In [18]:
# ------------------------------------------------------------
# 17. Export validation artefacts
# ------------------------------------------------------------

record_validation_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_validation_detailed.csv",
    index=False
)

field_validation_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_field_validation.csv",
    index=False
)

missing_records.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_missing_records.csv",
    index=False
)

unsupported_records.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_hallucinated_records.csv",
    index=False
)

fully_correct_records.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_fully_correct_records.csv",
    index=False
)

label_only_discrepancies.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_label_only_discrepancies.csv",
    index=False
)

other_discrepancies.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_other_discrepancies.csv",
    index=False
)

all_discrepancies.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_discrepant_records.csv",
    index=False
)

field_error_summary_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_field_error_summary.csv",
    index=False
)

section_performance_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_section_performance.csv",
    index=False
)

overall_metrics_df.to_csv(
    OUTPUT_DIR
    / "D3_branch_C_overall_metrics.csv",
    index=False
)

with open(
    OUTPUT_DIR
    / "D3_branch_C_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Validation artefacts saved."
)

Validation artefacts saved.


In [19]:
# ------------------------------------------------------------
# 18. Final validation report
# ------------------------------------------------------------

final_report = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH_ID,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "label_only_discrepant_records":
        N_LABEL_ONLY,

    "discrepant_records_total":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    "unsupported_records":
        N_UNSUPPORTED,

    "completeness":
        round(
            completeness,
            4
        ),

    "record_precision_exact":
        round(
            record_precision,
            4
        ),

    "record_recall_exact":
        round(
            record_recall,
            4
        ),

    "record_f1_exact":
        round(
            record_f1,
            4
        ),

    "overall_field_accuracy":
        round(
            overall_field_accuracy,
            4
        ),

    "schema_validity":
        schema_validity,

    "normalisation_integrity_passed":
        representation_integrity[
            "normalisation_integrity_passed"
        ],

    "comparison_rules_frozen_from_branch_A":
        True
}

print(
    json.dumps(
        final_report,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D3",
  "branch": "C",
  "reference_records": 70,
  "extracted_records": 74,
  "aligned_records": 69,
  "fully_correct_records": 40,
  "label_only_discrepant_records": 29,
  "discrepant_records_total": 29,
  "missing_records": 1,
  "unsupported_records": 5,
  "completeness": 0.9857,
  "record_precision_exact": 0.5405,
  "record_recall_exact": 0.5714,
  "record_f1_exact": 0.5556,
  "overall_field_accuracy": 0.9048,
  "schema_validity": true,
  "normalisation_integrity_passed": true,
  "comparison_rules_frozen_from_branch_A": true
}


In [20]:
# ------------------------------------------------------------
# 19. Download validation artefacts
# ------------------------------------------------------------

for output_file in sorted(
    OUTPUT_DIR.iterdir()
):
    if output_file.is_file():
        print(
            "Downloading:",
            output_file.name
        )
        files.download(
            output_file
        )

Downloading: D3_branch_C_discrepant_records.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D3_branch_C_field_error_summary.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: D3_branch_C_validation_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>